In [ ]:
import pandas as pd
import re 
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go
import plotly.colors as pcolors
import stata_setup
stata_setup.config('/Applications/Stata 17/', 'se')
import html
from pystata import stata
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [2]:
# data directory
data_dir = r"/your path/Project - Preprint"

# Survey result analysis

In [ ]:
# open file
survey_df = pd.read_csv(os.path.join(data_dir, "preprint survey.csv"))
survey_df

In [ ]:
# delete the second and third row, and reset the index
survey_df = survey_df.drop([0, 1]).reset_index(drop=True)
#delete if after column 17 is all NaN
survey_df = survey_df.loc[~survey_df.iloc[:, 17:].isna().all(axis=1)].copy()

In [ ]:
# read item_map.xlsx
item_map = pd.read_excel(os.path.join(data_dir, "item_map.xlsx"))
# replace the column names in survey_df with the corresponding names in item_map "label" column
survey_df.columns = survey_df.columns.map(item_map.set_index('raw')['col_name'])

# regression analysis

## country cleaning

In [10]:
reg_df = survey_df.copy()
reg_df = reg_df.apply(
    lambda col: col.str.strip() if col.dtype == 'object' else col
)
email = reg_df["RecipientEmail"].astype(str)

reg_df["country_new"] = pd.Series(
    np.select(
        [
            reg_df["RecipientEmail"].str.endswith(".edu", na=False),
            reg_df["RecipientEmail"].str.endswith(".ca", na=False),
        ],
        ["USA", "Canada"],
        default=np.nan,
    ),
    index=reg_df.index
).replace("nan", np.nan)

# fallback using country column if email-based classification is missing
reg_df["country_new"] = reg_df["country_new"].fillna(
    reg_df["country"].map({"USA": "USA", "Canada": "Canada"}).fillna("Others")
)
reg_df["country_new"].value_counts(dropna=False)

country_new
USA       1471
Canada     287
Name: count, dtype: int64

## gender cleaning

In [11]:
# replace reg_df['gender_new'] as Others if reg_df['gender_new'] is not in ['Woman', 'Man']
reg_df['gender_new'] = reg_df['gender'].apply(lambda x: 'Others' if x not in ['Woman', 'Man'] else x)

## academic rank cleaning

In [12]:
reg_df['acade_rank'].value_counts(dropna=False)

acade_rank
Full professor, or equivalent           717
NaN                                     413
Associate professor, or equivalent      342
Assistant professor, or equivalent      115
Research faculty, or equivalent          73
Other academic rank, please tell us:     41
Postdoctoral researcher                  33
Student, seeking doctoral degree         16
Teaching faculty, or equivalent           8
Name: count, dtype: int64

In [13]:
# replace reg_df['acade_rank'] = 'Senior' if reg_df['acade_rank'] contains "Full professor" or "Associate professor"
reg_df['acade_rank'] = reg_df['acade_rank'].apply(lambda x: 'Senior' if isinstance(x, str) and ('Full professor' in x or 'Associate professor' in x) else x)
# replace reg_df['acade_rank'] = 'Junior' if reg_df['acade_rank'] contains "Assistant professor" or "Postdoctoral researcher" or "Student"
reg_df['acade_rank'] = reg_df['acade_rank'].apply(lambda x: 'Junior' if isinstance(x, str) and ('Assistant professor' in x or 'Postdoctoral researcher' in x or 'Student' in x) else x)
# if not in the list, replace with Others
reg_df['acade_rank'] = reg_df['acade_rank'].apply(lambda x: 'Others' if x not in ['Senior', 'Junior'] else x)

reg_df['acade_rank'].value_counts(dropna=False)

acade_rank
Senior    1059
Others     535
Junior     164
Name: count, dtype: int64

## discipline cleaning

In [ ]:
# 1) Define mapping 
field_map = {
    # Molecular_Cellular_Biology
    "Biochemistry": "Molecular_Cellular_Biology",
    "Cell Biology": "Molecular_Cellular_Biology",
    "Genetics": "Molecular_Cellular_Biology",
    "Genomics": "Molecular_Cellular_Biology",
    "Molecular Biology": "Molecular_Cellular_Biology",
    "Microbiology": "Molecular_Cellular_Biology",

    # Biomedical_Health_Sciences
    "Cancer Biology": "Biomedical_Health_Sciences",
    "Epidemiology": "Biomedical_Health_Sciences",
    "Immunology": "Biomedical_Health_Sciences",
    "Neuroscience": "Biomedical_Health_Sciences",
    "Pathology": "Biomedical_Health_Sciences",
    "Pharmacology and Toxicology": "Biomedical_Health_Sciences",
    "Physiology": "Biomedical_Health_Sciences",
    "Developmental Biology": "Biomedical_Health_Sciences",

    # Organismal_Ecology_Evolution
    "Animal Behavior and Cognition": "Organismal_Ecology_Evolution",
    "Ecology": "Organismal_Ecology_Evolution",
    "Evolutionary Biology": "Organismal_Ecology_Evolution",
    "Paleontology": "Organismal_Ecology_Evolution",
    "Plant Biology": "Organismal_Ecology_Evolution",
    "Zoology": "Organismal_Ecology_Evolution",

    # Quantitative_Engineering_Biology
    "Bioengineering": "Quantitative_Engineering_Biology",
    "Bioinformatics": "Quantitative_Engineering_Biology",
    "Biophysics": "Quantitative_Engineering_Biology",
    "Synthetic Biology": "Quantitative_Engineering_Biology",
    "Systems Biology": "Quantitative_Engineering_Biology",
}

# 2) Create reg_df['field_new']
reg_df["field_new"] = (
    reg_df["field"]
      .astype(str)
      .str.strip()
      .map(field_map)
      .fillna("Other_disciplines")
)
reg_df['field_new'].value_counts(dropna=False)

field_new
Other_disciplines                   954
Biomedical_Health_Sciences          418
Molecular_Cellular_Biology          188
Organismal_Ecology_Evolution        114
Quantitative_Engineering_Biology     84
Name: count, dtype: int64

In [ ]:
# create an email list where reg_df["field_new"] is Other_disciplines
email_list = reg_df[reg_df["field_new"] == "Other_disciplines"]["RecipientEmail"].tolist()
email_list
# save email_list to csv
pd.DataFrame(email_list, columns=["email"]).to_csv(os.path.join(data_dir, "email_list.csv"), index=False)
# read email_list.csv
email_list = pd.read_csv(os.path.join(data_dir, "email_list.csv"))
# open df_email_titles_mapped.csv
df_email_titles_mapped = pd.read_csv(os.path.join(data_dir, "df_email_titles_mapped.csv"))

In [18]:
# set reg_df['field_new'] as df_email_titles_mapped['field_new'] if reg_df['Email']  = df_email_titles_mapped['Email']
m = df_email_titles_mapped.drop_duplicates("Email").set_index("Email")["field_new"]

mask = reg_df["field_new"].eq("Other_disciplines")

reg_df.loc[mask, "field_new"] = reg_df.loc[mask, "RecipientEmail"].map(m).combine_first(
    reg_df.loc[mask, "field_new"]
)
#delete rows where reg_df["field_new"] is Other_disciplines
reg_df = reg_df[reg_df["field_new"] != "Other_disciplines"]
reg_df["field_new"].value_counts(dropna=False)

field_new
Biomedical_Health_Sciences          1029
Quantitative_Engineering_Biology     270
Molecular_Cellular_Biology           239
Organismal_Ecology_Evolution         219
Name: count, dtype: int64

In [19]:
#gen dummy column for acade_rank_new, gender_new, country_new
reg_df = pd.get_dummies(
    reg_df,
    columns=["acade_rank", "gender_new", "country_new", "field_new"],
    drop_first=False,
    dtype=int
)

In [ ]:
# 1) Mappings
scale_1 = {"Not at all": 1, "A little": 2, "Somewhat": 3, "Very": 4, "Extremely": 5}
scale_2 = {"Not at all": 1, "Slightly": 2, "Somewhat": 3, "Very": 4, "Extremely": 5}
scale_3 = {"Poor": -2, "Fair": -1, "Good": 0, "Very good": 1, "Excellent": 2, "NA": np.nan}
scale_4 = {"Very negative": -2, "Somewhat negative": -1, "Neutral": 0, "Somewhat positive": 1, "Very positive": 2}
scale_5 = {"Not at all": 1, "A little": 2, "Somewhat": 3, "Quite a bit": 4, "A great deal": 5}
scale_6 = {"Never": 1, "Rarely": 2, "Sometimes": 3, "Very often": 4, "Extremely often": 5}

# frequency bins
scale_freq = {
    "Zero": 0,
    "1 to 5": 1,
    "6 to 10": 2,
    "11 to 15": 3,
    "More than 15": 4,
}

# binary
scale_yesno = {"No": 0, "Yes": 1}

# grant_included
scale_grant = {
    "No": 0,
    "Yes": 1,
    "I didn't submit any grant applications": np.nan, 
}

def normalize_by_contains(series: pd.Series, rules: list[tuple[str, str]]) -> pd.Series:
    s = series.copy()

    # only touch non-missing entries
    mask = s.notna()
    s.loc[mask] = s.loc[mask].astype(str).str.strip()

    for pattern, replacement in rules:
        s.loc[mask & s.astype(str).str.contains(pattern, case=False, na=False, regex=True)] = replacement

    # fix literal "nan"/"NaN"
    s = s.replace({"NaN": np.nan, "nan": np.nan})
    return s

contains_rules_credit = [
    (r"less credit", "Less credit"),
    (r"more credit", "More credit"),
    (r"no credit", "No credit"),
    (r"same credit", "Same credit"),
    (r"case by case", "Case by case"),
]

scale_credit = {
    "No credit": 1,
    "Less credit": 2,
    "Same credit": 3,
    "More credit": 4,
    "Case by case": np.nan
}

# Candidate scales to auto-detect
SCALES = {
    "scale_1": scale_1,
    "scale_2": scale_2,
    "scale_3": scale_3,
    "scale_4": scale_4,
    "scale_5": scale_5,
    "scale_6": scale_6,
    "scale_freq": scale_freq,
    "scale_yesno": scale_yesno,
    "scale_grant": scale_grant,
    "scale_credit": scale_credit
}

COLUMNS_NEED_CONTAINS = ["grant_credit_received", "grant_credit_should", "grant_review_credit",
    "hiring_credit_job", "hiring_credit_tenure", "hiring_credit_should"]


CONTAINS_RULES_BY_COL = {
    "grant_credit_received": contains_rules_credit,
    "grant_credit_should": contains_rules_credit,
    "grant_review_credit": contains_rules_credit,
    "hiring_credit_job": contains_rules_credit,
    "hiring_credit_tenure": contains_rules_credit,
    "hiring_credit_should": contains_rules_credit
}

for col in reg_df.columns:
    if not pd.api.types.is_object_dtype(reg_df[col]):
        continue

    # normalize by contains rules if this col needs it
    if col in COLUMNS_NEED_CONTAINS:
        reg_df[col] = normalize_by_contains(reg_df[col], CONTAINS_RULES_BY_COL[col])


# 2) Helper for cleaning unique values
def _nonmissing_unique(series: pd.Series) -> set:
    """
    Return unique non-missing string values.
    Handles both true NaN and literal strings like 'NaN'.
    """
    s = series.dropna().astype(str).str.strip()
    s = s[s.str.lower() != "nan"]  # sometimes NaN appears as string
    return set(s.unique())


def get_scale_mapping(series: pd.Series):
    """
    Detect which scale fits this series.
    Rule: all observed non-missing values must be a subset of the scale keys.
    """
    unique_vals = _nonmissing_unique(series)
    if len(unique_vals) == 0:
        return None

    for name, mapping in SCALES.items():
        if unique_vals.issubset(mapping.keys()):
            return mapping

    return None

# 3) Apply mapping to all object columns
for col in reg_df.columns:
    if not pd.api.types.is_object_dtype(reg_df[col]):
        continue

    mapping = get_scale_mapping(reg_df[col])
    if mapping is None:
        continue

    reg_df[col] = (
        reg_df[col]
        .astype(str).str.strip()
        .replace({"NaN": np.nan, "nan": np.nan})
        .map(mapping)
    )

In [23]:
# rename "field_new_Molecular_Cellular_Biology", "field_new_Organismal_Ecology_Evolution", "field_new_Quantitative_Engineering_Biology"
reg_df.rename(columns={"field_new_Molecular_Cellular_Biology": "Molecular_Cellular_Biology", "field_new_Organismal_Ecology_Evolution": "Organ_Eco_Evolution", "field_new_Quantitative_Engineering_Biology": "Quan_Engineer_Biology"}, inplace=True)

## run regression

In [ ]:
independents = "acade_rank_Junior acade_rank_Others gender_new_Others gender_new_Woman country_new_Canada Molecular_Cellular_Biology Organ_Eco_Evolution Quan_Engineer_Biology"

outcomes =[]
# if column index is from 17 to 90 and the value type is not string, then add the column name to outcomes
for i in range(17, 104):
    if not pd.api.types.is_string_dtype(reg_df.iloc[:, i]):
        outcomes.append(reg_df.columns[i])

outcomes

['popular',
 'overall_trust',
 'impact_accelerate',
 'impact_open_access',
 'impact_feedback',
 'impact_collab',
 'impact_eval_transform',
 'impact_reduce_trad',
 'neg_waste_time',
 'neg_misuse',
 'neg_llm_integrity',
 'neg_cognition',
 'neg_public_trust',
 'read_preprints_monthly',
 'read_reason_popularity',
 'read_reason_timeliness',
 'read_reason_authors',
 'read_reason_metrics',
 'read_reason_trust',
 'noread_unpopularity',
 'noread_untimeliness',
 'noread_authors',
 'noread_metrics',
 'noread_quality',
 'qual_design',
 'qual_theory',
 'qual_lit_review',
 'qual_conclusion',
 'qual_innovation',
 'qual_significance',
 'qual_ethics',
 'qual_materials',
 'qual_readability',
 'cited_preprints',
 'cite_reason_popularity',
 'cite_reason_timeliness',
 'cite_reason_authors',
 'cite_reason_metrics',
 'cite_reason_ethics',
 'cite_reason_materials',
 'cite_reason_trust',
 'cite_reason_requirement',
 'nocite_unpopularity',
 'nocite_untimeliness',
 'nocite_authors',
 'nocite_metrics',
 'nocite_e

In [25]:
outcome_results = []

for outcome in outcomes:
    print(outcome)

    if not pd.api.types.is_numeric_dtype(reg_df[outcome]):
        print(f"  → skipped (non-numeric dtype: {reg_df[outcome].dtype})")
        continue

    t = reg_df.copy()

    command = f"ologit {outcome} {independents}, or"
    stata.run("clear")
    stata.pdataframe_to_data(t)
    stata.run(command)

    # collect results
    results = stata.get_return()["r(table)"].T
    results = pd.DataFrame(results).iloc[[0, 1, 2, 3, 4, 5, 6, 7], :6]
    results.columns = ["odds_ratio", "se", "z", "p", "ci_low", "ci_high"]
    results["outcome"] = outcome
    results["var"] = ["acade_rank_Junior", "acade_rank_Others", "gender_new_Others", "gender_new_Woman", "country_new_Canada", "Molecular_Cellular_Biology", "Organ_Eco_Evolution", "Quan_Engineer_Biology"]

    outcome_results.append(results)


popular

Iteration 0:   log likelihood = -2516.4256  
Iteration 1:   log likelihood = -2474.6941  
Iteration 2:   log likelihood = -2474.5243  
Iteration 3:   log likelihood = -2474.5243  

Ordered logistic regression                             Number of obs =  1,752
                                                        LR chi2(8)    =  83.80
                                                        Prob > chi2   = 0.0000
Log likelihood = -2474.5243                             Pseudo R2     = 0.0167

------------------------------------------------------------------------------
     popular | Odds ratio   Std. err.      z    P>|z|     [95% conf. interval]
-------------+----------------------------------------------------------------
acade_~unior |   1.372274   .2097801     2.07   0.038     1.016991    1.851673
acade_rank~s |   .8569638   .1031273    -1.28   0.200     .6769063    1.084917
gender_new~s |   .8390513   .1191128    -1.24   0.216     .6352591     1.10822
gender_n~man |   .8

In [26]:
outcome_results_df = pd.concat(outcome_results)
# keep three decimal in p in outcome_results_df
outcome_results_df['p'] = outcome_results_df['p'].round(3)
# to csv
outcome_results_df.to_csv(os.path.join(data_dir, "ordinal_outcome_results_df.csv"), index=False)
outcome_results_df

,odds_ratio,se,z,p,ci_low,ci_high,outcome,var
0,1.372274,0.209780,2.070179,0.038,1.016991,1.851673,popular,acade_rank_Junior
1,0.856964,0.103127,-1.282692,0.200,0.676906,1.084917,popular,acade_rank_Others
2,0.839051,0.119113,-1.236137,0.216,0.635259,1.108220,popular,gender_new_Others
3,0.807359,0.082769,-2.087298,0.037,0.660393,0.987031,popular,gender_new_Woman
4,1.008647,0.118365,0.073367,0.942,0.801401,1.269487,popular,country_new_Canada
...,...,...,...,...,...,...,...,...
3,1.002547,0.292485,0.008720,0.993,0.565944,1.775973,hiring_credit_should,gender_new_Woman
4,1.062931,0.366620,0.176942,0.860,0.540646,2.089762,hiring_credit_should,country_new_Canada
5,1.803337,0.727511,1.461582,0.144,0.817864,3.976241,hiring_credit_should,Molecular_Cellular_Biology
6,1.943584,0.924556,1.396970,0.162,0.765050,4.937609,hiring_credit_should,Organ_Eco_Evolution


# Figures

In [ ]:
# Mappings
scale_1 = {"Not at all": 1, "A little": 2, "Somewhat": 3, "Very": 4, "Extremely": 5}
scale_2 = {"Not at all": 1, "Slightly": 2, "Somewhat": 3, "Very": 4, "Extremely": 5}
scale_3 = {"Poor": -2, "Fair": -1, "Good": 0, "Very good": 1, "Excellent": 2, "NA": np.nan}
scale_4 = {"Very negative": -2, "Somewhat negative": -1, "Neutral": 0, "Somewhat positive": 1, "Very positive": 2}

# Function to check if all non-null unique values are in a scale
def get_scale_mapping(series):
    unique_vals = set(series.dropna().astype(str).unique())
    if unique_vals.issubset(scale_1.keys()):
        return scale_1
    elif unique_vals.issubset(scale_2.keys()):
        return scale_2
    elif unique_vals.issubset(scale_3.keys()):
        return scale_3
    elif unique_vals.issubset(scale_4.keys()):
        return scale_4
    return None

# Bootstrap function to calculate confidence intervals
def bootstrap_ci(data, n_bootstrap=2000, confidence=0.95, random_state=None):

    if random_state is not None:
        np.random.seed(random_state)
    
    # Remove NaN values
    data_clean = data.dropna().values
    n = len(data_clean)
    
    if n == 0:
        return np.nan, np.nan, np.nan
    
    # Calculate actual mean
    mean = np.mean(data_clean)
    
    # Bootstrap resampling
    bootstrap_means = []
    for _ in range(n_bootstrap):
        # Resample with replacement
        bootstrap_sample = np.random.choice(data_clean, size=n, replace=True)
        bootstrap_means.append(np.mean(bootstrap_sample))
    
    # Calculate confidence interval using percentiles
    alpha = 1 - confidence
    ci_low = np.percentile(bootstrap_means, 100 * alpha / 2)
    ci_high = np.percentile(bootstrap_means, 100 * (1 - alpha / 2))
    
    return mean, ci_low, ci_high

# Generate the summary
results = []

for col in survey_df.columns:
    series = survey_df[col]

    # Skip non-object or numeric columns, which can't be mapped
    if not pd.api.types.is_object_dtype(series):
        continue

    scale = get_scale_mapping(series)
    if scale is None:
        continue  # Skip columns that don't match any scale

    # Map values to numeric
    mapped = series.astype(str).replace("nan", np.nan).map(scale)

    # Calculate bootstrap confidence intervals
    mean, ci_low, ci_high = bootstrap_ci(mapped, n_bootstrap=2000, confidence=0.95)

    results.append({
        "variable": col,
        "mean": mean,
        "std": mapped.std(),
        "n": mapped.count(),
        "ci_low": ci_low,
        "ci_high": ci_high
    })

# Create result dataframe
summary_df = pd.DataFrame(results)
summary_df[['mean', 'std', 'n', 'ci_low', 'ci_high']] = summary_df[['mean', 'std', 'n', 'ci_low', 'ci_high']].round(3)

/var/folders/v7/p_bgq74d2xz917nqhtz0dd4w0000gn/T/ipykernel_81733/4041641679.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mapped = series.astype(str).replace("nan", np.nan).map(scale)
/var/folders/v7/p_bgq74d2xz917nqhtz0dd4w0000gn/T/ipykernel_81733/4041641679.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mapped = series.astype(str).replace("nan", np.nan).map(scale)
/var/folders/v7/p_bgq74d2xz917nqhtz0dd4w0000gn/T/ipykernel_81733/4041641679.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will

,variable,mean,std,n,ci_low,ci_high
0,RecipientLastName,NaN,NaN,0,NaN,NaN
1,RecipientFirstName,NaN,NaN,0,NaN,NaN
2,ExternalReference,NaN,NaN,0,NaN,NaN
3,impact_accelerate,3.197,1.191,1663,3.138,3.252
4,impact_open_access,3.348,1.254,1662,3.288,3.409
...,...,...,...,...,...,...
67,dev_social_impact,0.211,0.905,1276,0.162,0.259
68,dev_career,-0.023,0.934,1248,-0.075,0.027
69,dev_funding,0.149,0.938,1282,0.098,0.200
70,dev_pub_opportunity,0.052,0.909,1290,-0.001,0.101


In [ ]:
# strip the value of variable column
summary_df['variable'] = summary_df['variable'].str.strip()
summary_df.to_excel(os.path.join(data_dir, "survey_summary.xlsx"), index=False)

In [ ]:
# Filter summary_df for specified variables
target_vars = [
    'read_reason_popularity',
    'read_reason_timeliness',
    'read_reason_authors',
    'read_reason_metrics',
    'read_reason_trust',
    'cite_reason_popularity',
    'cite_reason_timeliness',
    'cite_reason_authors',
    'cite_reason_metrics',
    'cite_reason_ethics',
    'cite_reason_materials',
    'cite_reason_trust',
    'cite_reason_requirement',
    'submit_open_reviews',
    'submit_history',
    'submit_visibility',
    'submit_citations',
    'submit_recommendation',
    'submit_dissemination',
    'submit_open_access',
    'submit_disclose'
]

# Filter and prepare data
plot_df = summary_df[summary_df['variable'].isin(target_vars)].copy()

# Map variables to categories
def get_category(var_name):
    if var_name.startswith('read'):
        return 'reading'
    elif var_name.startswith('cite'):
        return 'citing'
    elif var_name.startswith('submit'):
        return 'submitting'
    return 'other'

plot_df['category'] = plot_df['variable'].apply(get_category)

# Create clean labels for each metric (remove prefix and replace underscores)
plot_df['label'] = plot_df['variable'].str.replace('read_reason_', '', regex=False)
plot_df['label'] = plot_df['label'].str.replace('cite_reason_', '', regex=False)
plot_df['label'] = plot_df['label'].str.replace('submit_', '', regex=False)
plot_df['label'] = plot_df['label'].str.capitalize()

# Calculate error bar values (distance from mean to CI bounds)
plot_df['error_low'] = plot_df['mean'] - plot_df['ci_low']
plot_df['error_high'] = plot_df['ci_high'] - plot_df['mean']

# Define colors for each category
colors = {
    'reading': '#80C1C4',
    'citing': '#B2D3A4',
    'submitting': '#F9AE78'
}


fig = make_subplots(
    rows=3,
    cols=1,
    subplot_titles=('Reading', 'Citing', 'Submitting'),
    horizontal_spacing=0.08,
    vertical_spacing=0.1,
    shared_xaxes=True
)

global_span = plot_df['mean'].max() - plot_df['mean'].min()
offset = global_span * 0.05  # define once

for idx, category in enumerate(['reading', 'citing', 'submitting'], start=1):

    cat_df = plot_df[plot_df['category'] == category].copy()
    cat_df = cat_df.sort_values('mean', ascending=False)

    for _, row in cat_df.iterrows():
        fig.add_trace(
            go.Scatter(
                y=[row['label']],
                x=[row['mean']],
                mode='markers',
                marker=dict(
                    size=6,
                    opacity=0.8,
                    color=colors[category]
                ),
                error_x=dict(
                    type='data',
                    symmetric=False,
                    array=[row['ci_high'] - row['mean']],
                    arrayminus=[row['mean'] - row['ci_low']],
                    color=colors[category],
                    thickness=2
                ),
                showlegend=False
            ),
            row=idx, 
            col=1   
        )

        # text label
        fig.add_trace(
            go.Scatter(
                y=[row['label']],
                x=[row['ci_high'] + offset],
                mode='text',
                text=[f"{row['mean']:.3f}"],
                textfont=dict(size=9),
                showlegend=False
            ),
            row=idx,
            col=1
        )

    # y-axis formatting per row
    fig.update_yaxes(
        tickfont=dict(size=11),
        row=idx,
        col=1
    )

# put x-axis title only on bottom subplot
fig.update_xaxes(
    title_text="Mean",
    tickfont=dict(size=12),
    title_font=dict(size=12),
    row=3,
    col=1
)
fig.update_yaxes(
    tickfont=dict(size=12),
    title_font=dict(size=12)
)

fig.update_layout(
    height=700,    
    width=400,
    template="simple_white",
    font=dict(size=12, family="Arial"),
    margin=dict(l=100, r=60, b=80, t=120)
)

for ann in fig.layout.annotations:
    ann.font.size = 12
    ann.y += 0  # smaller shift since vertical layout

fig.show()

# write as PDF
fig.write_image("read_cite_submit.pdf")

In [ ]:
# Filter summary_df for specified variables
target_vars = [
'noread_unpopularity',
'noread_untimeliness',
'noread_authors',
'noread_metrics',
'noread_quality',
'nocite_unpopularity',
'nocite_untimeliness',
'nocite_authors',
'nocite_metrics',
'nocite_quality',
]

# Filter and prepare data
plot_df = summary_df[summary_df['variable'].isin(target_vars)].copy()

# Map variables to categories
def get_category(var_name):
    if var_name.startswith('noread'):
        return 'not reading'
    elif var_name.startswith('nocite'):
        return 'not citing'

plot_df['category'] = plot_df['variable'].apply(get_category)

# Create clean labels for each metric and capitalize the first letter
plot_df['label'] = plot_df['variable'].str.replace('noread_', '', regex=False)
plot_df['label'] = plot_df['label'].str.replace('nocite_', '', regex=False)
plot_df['label'] = plot_df['label'].str.capitalize()

# Calculate error bar values (distance from mean to CI bounds)
plot_df['error_low'] = plot_df['mean'] - plot_df['ci_low']
plot_df['error_high'] = plot_df['ci_high'] - plot_df['mean']

# Define colors for each category
colors = {
    'not reading': '#80C1C4',
    'not citing': '#B2D3A4',
}


# Prepare shared x-axis labels and  order by mean
metrics = (
    plot_df.query("category == 'not reading'")
    .sort_values("mean", ascending=True)   # order by mean
    ["label"]
    .tolist()
)

fig = make_subplots(
    rows=1,
    cols=1,
    shared_yaxes=True
)


# Add bars for each category
for category in ["not reading", "not citing"]:

    sub = plot_df[plot_df["category"] == category].copy()

    # Ensure alignment with shared x labels
    sub = sub.set_index("label").reindex(metrics)

    fig.add_trace(
        go.Bar(
            x=metrics,
            y=sub["mean"],
            name=category.capitalize(),
            marker=dict(
                color=colors[category],
                opacity=0.85
            ),
            error_y=dict(
                type="data",
                symmetric=False,
                array=sub["ci_high"] - sub["mean"],
                arrayminus=sub["mean"] - sub["ci_low"],
                thickness=1.8,
                color="rgba(0,0,0,0.4)"
            )
        )
    )

fig.update_layout(
    barmode="group",              
    bargap=0.25,
    bargroupgap=0.08,
    height=420,
    width=460,
    template="simple_white",
    font=dict(size=11, family="Arial"),
    legend=dict(
        orientation="v",      # vertical legend
        x=0.01,               # left
        y=0.99,               # top
        xanchor="left",
        yanchor="top",
        font=dict(size=12)
    ),
    margin=dict(l=59, r=59, b=160, t=80)
)

fig.update_yaxes(
    title_text="Mean",
    title_font=dict(size=12)
)

fig.update_xaxes(
    tickangle=0,
    tickfont=dict(size=11)
)

fig.show()

# write as PDF
fig.write_image("No_read_cite.pdf")

In [ ]:
# Filter summary_df for specified variables
target_vars = [
'qual_design',
'qual_theory',
'qual_lit_review',
'qual_conclusion',
'qual_innovation',
'qual_significance',
'qual_ethics',
'qual_materials',
'qual_readability'
]

# Filter and prepare data
plot_df = summary_df[summary_df['variable'].isin(target_vars)].copy()


# create a label map
# Define label mapping
label_map = {
    'qual_design': 'Design',
    'qual_theory': 'Theory',
    'qual_lit_review': 'Literature review',
    'qual_conclusion': 'Conclusion',
    'qual_innovation': 'Innovation',
    'qual_significance': 'Significance',
    'qual_ethics': 'Ethics',
    'qual_materials': 'Materials',
    'qual_readability': 'Readability'
}

# Filter and prepare data
plot_df = summary_df[summary_df['variable'].isin(target_vars)].copy()

# Create label column
plot_df['label'] = plot_df['variable'].map(label_map)

# Calculate error bar values (distance from mean to CI bounds)
plot_df['error_low'] = plot_df['mean'] - plot_df['ci_low']
plot_df['error_high'] = plot_df['ci_high'] - plot_df['mean']

# Define colors for each category
colors = {
    'quality': '#94b5d7'
}

plot_df = plot_df.sort_values("mean", ascending=True) 

# Create figure
fig = make_subplots(rows=1, cols=1)

fig.add_trace(
    go.Bar(
        y=plot_df["label"],          # ← categories on Y
        x=plot_df["mean"],           # ← values on X
        orientation="h",             # ← horizontal bars
        name="Quality",
        marker=dict(
            color=colors["quality"],
            opacity=0.85
        ),
        error_x=dict(          
            type="data",
            symmetric=False,
            array=plot_df["ci_high"] - plot_df["mean"],
            arrayminus=plot_df["mean"] - plot_df["ci_low"],
            thickness=1.8,
            color="rgba(0,0,0,0.4)"  
        )
    )
)

# Layout & formatting
fig.update_layout(
    height=350,
    width=450,
    template="simple_white",
    font=dict(size=11, family="Arial"),
    margin=dict(l=160, r=60, b=60, t=60)
)

fig.update_xaxes(
    title_text="Mean",
    title_font=dict(size=12)
)

fig.update_yaxes(
    title_text="",
    tickfont=dict(size=12),
    autorange="reversed"   
)

fig.show()

# write as PDF
fig.write_image("quality.pdf")

In [ ]:
# Filter summary_df for specified variables
target_vars = [
'dev_collab',
'dev_citations',
'dev_social_impact',
'dev_career',
'dev_funding',
'dev_pub_opportunity'
]

# Filter and prepare data
plot_df = summary_df[summary_df['variable'].isin(target_vars)].copy()
print(plot_df)

# create a label map
# Define label mapping
label_map = {
    'dev_collab': 'Collaborations',
    'dev_citations': 'Citations',
    'dev_social_impact': 'Social impact',
    'dev_career': 'Career advancement',
    'dev_funding': 'Funding opportunities',
    'dev_pub_opportunity': 'Publication opportunities'
}

# Filter and prepare data
plot_df = summary_df[summary_df['variable'].isin(target_vars)].copy()

# Create label column
plot_df['label'] = plot_df['variable'].map(label_map)

# Calculate error bar values (distance from mean to CI bounds)
plot_df['error_low'] = plot_df['mean'] - plot_df['ci_low']
plot_df['error_high'] = plot_df['ci_high'] - plot_df['mean']

# Define colors for each category
colors = {
    'development': '#cabad7'
}

plot_df = plot_df.sort_values("mean", ascending=False) 

-
# Create figure
-
fig = make_subplots(rows=1, cols=1)

fig.add_trace(
    go.Bar(
        y=plot_df["label"],          # ← categories on Y
        x=plot_df["mean"],           # ← values on X
        orientation="h",             # ← horizontal bars
        name="Development",
        marker=dict(
            color=colors["development"],
            opacity=0.85
        ),
        error_x=dict(          
            type="data",
            symmetric=False,
            array=plot_df["ci_high"] - plot_df["mean"],
            arrayminus=plot_df["mean"] - plot_df["ci_low"],
            thickness=1.8,
            color="rgba(0,0,0,0.4)" 
        )
    )
)

-
# Layout & formatting
-
fig.update_layout(
    height=300,
    width=450,
    template="simple_white",
    font=dict(size=12, family="Arial"),
    margin=dict(l=160, r=60, b=60, t=60)
)

fig.update_xaxes(
    title_text="Mean",
    title_font=dict(size=12)
)

fig.update_yaxes(
    title_text="",
    tickfont=dict(size=12)
)

fig.show()

# write as PDF
fig.write_image("career_dev.pdf")

               variable   mean    std     n  ci_low  ci_high
65           dev_collab  0.311  0.836  1288   0.263    0.356
66        dev_citations  0.238  0.957  1299   0.185    0.289
67    dev_social_impact  0.211  0.905  1276   0.162    0.259
68           dev_career -0.023  0.934  1248  -0.075    0.027
69          dev_funding  0.149  0.938  1282   0.098    0.200
70  dev_pub_opportunity  0.052  0.909  1290  -0.001    0.101


In [ ]:
# Filter summary_df for specified variables
target_vars = [
'nosubmit_recognition',
'nosubmit_discouraged',
'nosubmit_reputation',
'nosubmit_plagiarism',
'nosubmit_perception',
'nosubmit_collaborators',
'nosubmit_unaware'
]

# Filter and prepare data
plot_df = summary_df[summary_df['variable'].isin(target_vars)].copy()


# create a label map
# Define label mapping
label_map = dict(zip(item_map["col_name"], item_map["label"]))

# Filter and prepare data
plot_df = summary_df[summary_df['variable'].isin(target_vars)].copy()

# Create label column
plot_df['label'] = plot_df['variable'].map(label_map)

# Calculate error bar values (distance from mean to CI bounds)
plot_df['error_low'] = plot_df['mean'] - plot_df['ci_low']
plot_df['error_high'] = plot_df['ci_high'] - plot_df['mean']

# Define colors for each category
colors = {
    'nosubmit': '#C1D09D'
}

plot_df["variable"] = pd.Categorical(
    plot_df["variable"],
    categories=target_vars,
    ordered=True
)
plot_df = plot_df.sort_values("mean", ascending=False) 

-
# Create figure
-
fig = make_subplots(rows=1, cols=1)

fig.add_trace(
    go.Bar(
        y=plot_df["label"],          # ← categories on Y
        x=plot_df["mean"],           # ← values on X
        orientation="h",             # ← horizontal bars
        marker=dict(
            color=colors["nosubmit"],
            opacity=0.85
        ),
        error_x=dict(          
            type="data",
            symmetric=False,
            array=plot_df["ci_high"] - plot_df["mean"],
            arrayminus=plot_df["mean"] - plot_df["ci_low"],
            thickness=1.8,
            color="rgba(0,0,0,0.4)" 
        )
    )
)

-
# Layout & formatting
-
fig.update_layout(
    height=300,
    width=450,
    template="simple_white",
    font=dict(size=12, family="Arial"),
    margin=dict(l=160, r=60, b=60, t=60)
)

fig.update_yaxes(
    tickfont=dict(size=12)
)

fig.update_xaxes(
    tickfont=dict(size=12),
    title_text="Mean",
    title_font=dict(size=12)
)

fig.show()

# write as PDF
fig.write_image("not_submit_reason.pdf")

In [38]:
# Get columns from the 18th to the last
target_columns = survey_df.columns[17:]

# Exclude columns containing '_text' and 'final_comments'
filtered_columns = [col for col in target_columns if '_text' not in col and col != 'final_comments']

result = []

for col in filtered_columns:
    dist = survey_df[col].value_counts(normalize=True, dropna=False).reset_index()
    dist.columns = ['value', 'percentage']
    dist['variable'] = col
    result.append(dist)

# Combine all into one DataFrame
distribution_df = pd.concat(result, ignore_index=True)[['variable', 'value', 'percentage']]

In [39]:
#show the rows where variable in "gender", 'sector', 'acade_rank', 'country'
distribution_df.to_excel(r"/Users/hongxi/Desktop/Project - Preprint/distribution_df.xlsx", index=False)
distribution_df

,variable,value,percentage
0,popular,Somewhat,0.383390
1,popular,A little,0.252560
2,popular,Very,0.188851
3,popular,Not at all,0.124005
4,popular,Extremely,0.048350
...,...,...,...
571,field,Systems Biology,0.003982
572,field,Mathematics,0.002275
573,field,Computer science,0.002275
574,field,Paleontology,0.001706


In [40]:
# get plotly default colors
def generate_color_list(num_colors):
    # Get a list of colors from the Plotly color scales
    color_scale = pcolors.DEFAULT_PLOTLY_COLORS
    colors = color_scale * (num_colors // len(color_scale) + 1)

    color_list = []
    # Modify the colors to include the desired opacity
    for i in range(num_colors):
        color_list.append(f"rgba{colors[i][3:-1]}" + ", {})")
    return color_list

color_list = generate_color_list(9)
color_list

['rgba(31, 119, 180, {})',
 'rgba(255, 127, 14, {})',
 'rgba(44, 160, 44, {})',
 'rgba(214, 39, 40, {})',
 'rgba(148, 103, 189, {})',
 'rgba(140, 86, 75, {})',
 'rgba(227, 119, 194, {})',
 'rgba(127, 127, 127, {})',
 'rgba(188, 189, 34, {})']

In [56]:
# given a column list: [grant_credit_received, grant_credit_should, grant_review_preprint_freq, grant_review_credit,
# examined_preprints_freq, hiring_credit_job, hiring_credit_tenure, hiring_credit_should] in distribution_df[variable],
# generate a new dataframe for each variable by only keep rows with not null distribution_df[values] and recalculate percentage by adding up all percentages as denominator and
# each original percentage as numerator
variables = [
   # "grant_credit_received", 
    "grant_credit_should", #"grant_review_preprint_freq",
    "grant_review_credit", #"examined_preprints_freq", 
    "hiring_credit_job", "hiring_credit_tenure", "hiring_credit_should"
]

# build recalculated dataframes, then concatenate into one
result_df = pd.concat([
    (
        distribution_df.loc[
            (distribution_df["variable"] == var) & (distribution_df["value"].notna())
        ].assign(
            percentage=lambda d: d["percentage"] / d["percentage"].sum()
        )
    )
    for var in variables
], ignore_index=True)

result_df

,variable,value,percentage
0,grant_credit_should,Less credit,0.496256
1,grant_credit_should,Case by case,0.244384
2,grant_credit_should,No credit,0.147720
3,grant_credit_should,Same credit,0.104833
4,grant_credit_should,More credit,0.006807
5,grant_review_credit,Less credit,0.498382
6,grant_review_credit,Case by case,0.266181
7,grant_review_credit,Same credit,0.120550
8,grant_review_credit,No credit,0.112460
9,grant_review_credit,More credit,0.002427


In [57]:
variables = [
    "read_preprints_monthly", "cited_preprints", 
    "submitted_preprints", "grant_included",
    "grant_review_preprint_freq", "examined_preprints_freq" 
]

# build recalculated dataframes, then concatenate into one
result_df_2 = pd.concat([
    (
        distribution_df.loc[
            (distribution_df["variable"] == var) & (distribution_df["value"].notna())
        ].assign(
            percentage=lambda d: d["percentage"] / d["percentage"].sum()
        )
    )
    for var in variables
], ignore_index=True)

result_df_2["value"] = result_df_2["value"].replace({
    "11 to 15": "11 and more",
    "More than 15": "11 and more"
})

result_df_2["value"] = result_df_2["value"].replace({
    "Extremely often": "Often",
    "Very often ": "Often"
})

result_df_2 = (
    result_df_2
    .groupby(["variable", "value"], as_index=False)["percentage"]
    .sum()
)
# delete the row with value "I didn't submit any grant applications"
result_df_2 = result_df_2[result_df_2["value"] != "I didn't submit any grant applications"]

# recalculate the percentage of grant_included
mask = (result_df_2["variable"] == "grant_included") & (result_df_2["percentage"].notna())

result_df_2.loc[mask, "percentage"] = (
    result_df_2.loc[mask, "percentage"] /
    result_df_2.loc[mask, "percentage"].sum()
)

result_df_2

,variable,value,percentage
0,cited_preprints,No,0.692943
1,cited_preprints,Yes,0.307057
2,examined_preprints_freq,Never,0.253369
3,examined_preprints_freq,Often,0.138365
4,examined_preprints_freq,Rarely,0.278527
5,examined_preprints_freq,Sometimes,0.329739
7,grant_included,No,0.729951
8,grant_included,Yes,0.270049
9,grant_review_preprint_freq,Never,0.356856
10,grant_review_preprint_freq,Often,0.071371


In [59]:

# 1. Prepare labels

label_map = dict(zip(
    item_map["col_name"].str.strip(),
    item_map["label"].str.strip()
))


# 2. Clean & normalize data

df = result_df.copy()
df = df.dropna(subset=["value"])
df["value"] = df["value"].str.strip()
df = df[df["value"] != ""]
# Combine Same credit + More credit
df["value"] = df["value"].replace({
    "Same credit": "Same/More credit",
    "More credit": "Same/More credit"
})

# Re-aggregate after collapsing categories
df = (
    df.groupby(["variable", "value"], as_index=False)["percentage"]
      .sum()
)

# Normalize percentages within each variable
df["percentage"] = (
    df["percentage"] /
    df.groupby("variable")["percentage"].transform("sum")
)


# 3. Determine global value order

value_levels = ["No credit", "Less credit", "Case by case", "Same/More credit"]

# 4. Color palette

base_colors = [
    "#ed8687",
    "#feb29b",
    "#ffd19d",
    "#bed2c6",
    "#c9e0e5"
]

color_map = dict(zip(value_levels, base_colors[:len(value_levels)]))

x_label_map = {
    "grant_credit_should": "Non-grant <br>reviewers",
    "grant_review_credit": "Grant <br> reviewers",
    "hiring_credit_should": "Non-committee",
    "hiring_credit_job": "Hiring <br> committee",
    "hiring_credit_tenure": "Tenure <br> committee"
}
x_order = list(x_label_map.keys())


# 5. Build stacked bars

fig = go.Figure()

for val in value_levels:
    t = df[df["value"] == val]

    if t.empty:
        continue

    fig.add_bar(
        x=t["variable"],
        y=t["percentage"] * 100,
        name=val,
        marker_color=color_map[val],
        text=(t["percentage"] * 100).round(1).astype(str) + "%",
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(size=11),
        constraintext="none"
    )


# 6. Axis labels & layout

fig.update_layout(
    barmode="stack",
    template="simple_white",
    height=420,
    width=600,
    font=dict(size=12, family="Arial"),
    margin=dict(l=100, r=100, t=20, b=20),
    yaxis=dict(
        title="Percentage of respondents",
        range=[0, 100],
        ticksuffix="%"
    ),
    xaxis=dict(
        title="",
        type="category",
        categoryorder="array",
        categoryarray=x_order,
        tickvals=x_order,
        ticktext=[x_label_map[v] for v in x_order]
    ),

    legend=dict(
        orientation="h",
        x=0.5,
        xanchor="center",
        y=-0.2,
        yanchor="top"
    )
)
fig.update_xaxes(
    tickangle=0,
    tickfont=dict(size=12),
)

fig.update_yaxes(
    tickfont=dict(size=12),
    title_font=dict(size=12)
)
fig.show()

#save as PDF
fig.write_image("credit_freq.pdf")


In [60]:
# data
df = result_df_2.copy()
# 2. Orders
var_order = ["submitted_preprints", "read_preprints_monthly"]
value_levels = ["Zero", "1 to 5", "6 to 10", "11 and more"]

# (optional) nicer y labels
var_label_map = {
    "read_preprints_monthly": "Reading",
    "submitted_preprints": "Submitting"
}


# 3. Color palette
base_colors = [
    "#ed8687",
    "#feb29b",
    "#ffd19d",
    "#bed2c6",
    "#c9e0e5"
]
color_map = dict(zip(value_levels, base_colors[:len(value_levels)]))


# 4. Build horizontal stacked bars

fig = go.Figure()

for val in value_levels:
    t = df[df["value"] == val].copy()
    if t.empty:
        continue

    # enforce order for the two variables
    t["variable"] = pd.Categorical(t["variable"], categories=var_order, ordered=True)
    t = t.sort_values("variable")

    fig.add_bar(
        y=t["variable"].astype(str).map(var_label_map).fillna(t["variable"].astype(str)),
        x=t["percentage"] * 100,                                   
        name=val,
        marker_color=color_map[val],
        orientation="h",
        text=(t["percentage"] * 100).round(1).astype(str) + "%",
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(size=10),
        constraintext="none"
    )

# 5. Layout

fig.update_layout(
    barmode="stack",
    template="simple_white",
    height=260,
    width=720,
    font=dict(size=12, family="Arial"),
    margin=dict(l=130, r=150, t=90, b=80),

    xaxis=dict(
        title="Percentage of respondents",
        range=[0, 100],
        ticksuffix="%"
    ),

    yaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=[var_label_map.get(v, v) for v in var_order]
    ),

    legend=dict(
        orientation="v",
        y=0.5,
        yanchor="middle",
        x=1.02,
        xanchor="left"
    )
)

fig.show()
#save as PDF
fig.write_image("read_submit_freq.pdf")


In [ ]:

# 1. Clean & normalize data

df = result_df_2.copy()

# only keep these two variables
df = df[df["variable"].isin(["cited_preprints", "grant_included"])].copy()

df = df.dropna(subset=["value"])
df["value"] = df["value"].astype(str).str.strip()

# keep only Yes/No
df = df[df["value"].isin(["No", "Yes"])].copy()

# Normalize percentages within each variable (robust to NaN % rows)
df["percentage"] = (
    df["percentage"] /
    df.groupby("variable")["percentage"].transform(lambda s: s.dropna().sum())
)


# 2. Orders

var_order = ["grant_included", "cited_preprints"]
value_levels = ["No", "Yes"]

# nicer y labels
var_label_map = {
    "cited_preprints": "Citing",
    "grant_included": "Grant"
}


# 3. Color palette

base_colors = ["#feb29b", "#bed2c6"]   # No, Yes
color_map = dict(zip(value_levels, base_colors))


# 4. Build horizontal stacked bars

fig = go.Figure()

for val in value_levels:
    t = df[df["value"] == val].copy()
    if t.empty:
        continue

    # enforce order
    t["variable"] = pd.Categorical(t["variable"], categories=var_order, ordered=True)
    t = t.sort_values("variable")

    fig.add_bar(
        y=t["variable"].astype(str).map(var_label_map).fillna(t["variable"].astype(str)),
        x=t["percentage"] * 100,
        name=val,
        marker_color=color_map[val],
        orientation="h",
        text=(t["percentage"] * 100).round(1).astype(str) + "%",
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(size=11),
        constraintext="none"
    )


# 5. Layout

fig.update_layout(
    barmode="stack",
    template="simple_white",
    height=260,
    width=720,
    font=dict(size=12, family="Arial"),
    title="Distribution of preprint-related behaviors",
    margin=dict(l=130, r=150, t=90, b=80),

    xaxis=dict(
        title="Percentage of respondents",
        range=[0, 100],
        ticksuffix="%"
    ),

    yaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=[var_label_map.get(v, v) for v in var_order]
    ),

    legend=dict(
        orientation="v",
        y=0.5,
        yanchor="middle",
        x=1.02,
        xanchor="left",
        title=""
    )
)

fig.show()
#save as PDF
fig.write_image("cite_grant_freq.pdf")



In [ ]:

# 1. Clean & normalize data

df = result_df_2.copy()

# only keep these two variables
df = df[df["variable"].isin(["grant_review_preprint_freq", "examined_preprints_freq"])].copy()

# define final value order
value_levels = ["Never", "Rarely", "Sometimes", "Often"]

# keep only these 4 levels
df = df[df["value"].isin(value_levels)].copy()

# Normalize percentages within each variable
df["percentage"] = (
    df["percentage"] /
    df.groupby("variable")["percentage"].transform(lambda s: s.dropna().sum())
)


# 2. Orders

var_order = ["examined_preprints_freq", "grant_review_preprint_freq"]

# nicer y labels
var_label_map = {
    "grant_review_preprint_freq": "Grant review",
    "examined_preprints_freq": "Hiring and promoting"
}


# 3. Color palette

# 4-level ordinal palette (low -> high)
base_colors = [
    "#ed8687",  # Never
    "#feb29b",  # Rarely
    "#ffd19d",  # Sometimes
    "#bed2c6",  # Often
]
color_map = dict(zip(value_levels, base_colors))


# 4. Build horizontal stacked bars

fig = go.Figure()

for val in value_levels:
    t = df[df["value"] == val].copy()
    if t.empty:
        continue

    # enforce order
    t["variable"] = pd.Categorical(t["variable"], categories=var_order, ordered=True)
    t = t.sort_values("variable")

    fig.add_bar(
        y=t["variable"].astype(str).map(var_label_map).fillna(t["variable"].astype(str)),
        x=t["percentage"] * 100,
        name=val,
        marker_color=color_map[val],
        orientation="h",
        text=(t["percentage"] * 100).round(1).astype(str) + "%",
        textposition="inside",
        insidetextanchor="middle",
        textfont=dict(size=11),
        constraintext="none"
    )


# 5. Layout

fig.update_layout(
    barmode="stack",
    template="simple_white",
    height=260,
    width=720,
    font=dict(size=12, family="Arial"),
    margin=dict(l=130, r=150, t=90, b=80),

    xaxis=dict(
        title="Percentage of respondents",
        range=[0, 100],
        ticksuffix="%"
    ),

    yaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=[var_label_map.get(v, v) for v in var_order]
    ),

    legend=dict(
        orientation="v",
        y=0.5,
        yanchor="middle",
        x=1.02,
        xanchor="left",
        title=""
    )
)

fig.show()
#save as PDF
fig.write_image("evaluation_freq.pdf")


In [ ]:

# 1) Prepare labels + ordering

vars_to_plot = [
    "impact_accelerate",
    "impact_open_access",
    "impact_feedback",
    "impact_collab",
    "impact_eval_transform",
    "impact_reduce_trad"
]

label_map = dict(zip(item_map["col_name"], item_map["label"]))
label_map["impact_reduce_trad"] = "Reduced emphasis<br>on top-tier venues"
# keep the order exactly as vars_to_plot, but show labels
var_to_label = {v: label_map.get(v, v) for v in vars_to_plot}
y_labels = sorted(var_to_label.values()) # CHANGED: order variables by alphabetical label instead of original vars_to_plot order

value_order = ["Not at all", "A little", "Somewhat", "Very", "Extremely"]


# 2) Build one long dataframe

t = distribution_df.query("variable in @vars_to_plot").copy()
t = t.dropna(subset=["value"])
t["value"] = t["value"].astype(str).str.strip()
t = t[t["value"] != ""]

# normalize percentages *within each variable* ignoring missing categories
t["percentage"] = t["percentage"] / t.groupby("variable")["percentage"].transform("sum")

# keep value order
t["value"] = pd.Categorical(t["value"], categories=value_order, ordered=True)

# map variable -> y-axis label
t["var_label"] = t["variable"].map(label_map)

# keep variable order on y
t["var_label"] = pd.Categorical(t["var_label"], categories=y_labels, ordered=True)

# Convert to %
t["pct"] = t["percentage"] * 100


# 3) Build stacked bar figure

color_map = {
    "Not at all":  "#2d4f74",
    "A little":  "#99bbe0",
    "Somewhat":  "#d5e9f4",
    "Very":        "#cabad7",
    "Extremely":   "#9b9ac1",
}


fig = go.Figure()

for i, val in enumerate(value_order):
    tt = t[t["value"] == val].copy()
    #  enforce consistent y order in every trace
    tt = tt.sort_values("var_label")
    fig.add_trace(
        go.Bar(
            x=tt["pct"],
            y=tt["var_label"],
            name=val,
            orientation="h",
            marker=dict(color=color_map.get(val, "#cccccc"), opacity=0.9),
            text=tt["pct"].round(1).astype(str) + "%",
            textposition="inside",
            insidetextanchor="middle",
            hovertemplate="<b>%{y}</b><br>" + val + ": %{x:.1f}%<extra></extra>",
        )
    )

fig.update_layout(
    barmode="stack",
    height=320,
    width=720,
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.02),
    template="simple_white",
    font=dict(size=12, family="Arial"),
    margin=dict(l=50, r=100, b=50, t=50),
)

fig.update_xaxes(
    title_text="Percentage of respondents",
    tickfont=dict(size=12),
    title_font=dict(size=12),
    range=[0, 100],
    ticksuffix="%",
    showgrid=False
)

fig.update_yaxes(
    title_text="",
    tickfont=dict(size=12),
    categoryorder="array",
    categoryarray=y_labels,
    autorange="reversed"
)

fig.write_image("positive_impact.pdf")
fig.show()

In [ ]:

# 1) Prepare labels + ordering

vars_to_plot = [
    "neg_waste_time",
    "neg_misuse",
    "neg_llm_integrity",
    "neg_cognition",
    "neg_public_trust"
]

label_map = dict(zip(item_map["col_name"], item_map["label"]))

# keep the order exactly as vars_to_plot, but show labels
var_to_label = {v: label_map.get(v, v) for v in vars_to_plot}
y_labels = sorted(var_to_label.values()) 
value_order = ["Not at all", "A little", "Somewhat", "Very", "Extremely"]


# 2) Build one long dataframe

t = distribution_df.query("variable in @vars_to_plot").copy()
t = t.dropna(subset=["value"])
t["value"] = t["value"].astype(str).str.strip()
t = t[t["value"] != ""]

# normalize percentages *within each variable* ignoring missing categories
t["percentage"] = t["percentage"] / t.groupby("variable")["percentage"].transform("sum")

# keep value order
t["value"] = pd.Categorical(t["value"], categories=value_order, ordered=True)

# map variable -> y-axis label
t["var_label"] = t["variable"].map(label_map)

# keep variable order on y
t["var_label"] = pd.Categorical(t["var_label"], categories=y_labels, ordered=True)

# Convert to %
t["pct"] = t["percentage"] * 100


# 3) Build stacked bar figure

color_map = {
    "Not at all":  "#2d4f74",
    "A little":  "#99bbe0",
    "Somewhat":  "#d5e9f4",
    "Very":        "#cabad7",
    "Extremely":   "#9b9ac1",
}


fig = go.Figure()

for i, val in enumerate(value_order):
    tt = t[t["value"] == val].copy()
    # enforce consistent y order in every trace
    tt = tt.sort_values("var_label")
    fig.add_trace(
        go.Bar(
            x=tt["pct"],
            y=tt["var_label"],
            name=val,
            orientation="h",
            marker=dict(color=color_map.get(val, "#cccccc"), opacity=0.9),
            text=tt["pct"].round(1).astype(str) + "%",
            textposition="inside",
            insidetextanchor="middle",
            hovertemplate="<b>%{y}</b><br>" + val + ": %{x:.1f}%<extra></extra>",
        )
    )

fig.update_layout(
    barmode="stack",
    height=320,
    width=720,
    legend=dict(orientation="v", yanchor="middle", y=0.5, xanchor="left", x=1.02),
    template="simple_white",
    font=dict(size=12, family="Arial"),
    margin=dict(l=50, r=100, b=50, t=50),
)

fig.update_xaxes(
    title_text="Percentage of respondents",
    tickfont=dict(size=12),
    title_font=dict(size=12),
    range=[0, 100],
    ticksuffix="%",
    showgrid=False
)

fig.update_yaxes(
    title_text="",
    categoryorder="array",
    categoryarray=y_labels,
    autorange="reversed",
    tickfont=dict(size=12)
)

fig.write_image("negative_impact.pdf")
fig.show()

# Free-text analysis

In [3]:
# Load results
df_2 = pd.read_excel("free_text_labeled_wide_original.xlsx")

themes = [
    "quality_of_preprints",
    "impact_on_science_and_society",
    "impact_on_scholars_professional_development"]

sentiments = ["negative", "neutral", "positive"]

# Total number of comments
N = len(df_2)

# Build Theme x Sentiment table
rows = []
for theme in themes:
    present_col = f"{theme}_present"
    sent_col = f"{theme}_sentiment"
    
    # only look at rows where the theme is present
    sub = df_2[df_2[present_col] == True]

    n_present = len(sub)
    # count sentiments within this theme
    counts = sub[sent_col].value_counts(dropna=False).to_dict()

    row = {
        "theme": theme,
        "n_present": n_present,
        "pct_of_all_comments": n_present / N if N else 0.0,
        "n_negative": counts.get("negative", 0),
        "n_neutral": counts.get("neutral", 0),
        "n_positive": counts.get("positive", 0),
    }

    # within-theme sentiment proportions
    denom = n_present if n_present > 0 else 1
    row.update({
        "pct_negative_within_theme": row["n_negative"] / denom,
        "pct_neutral_within_theme": row["n_neutral"] / denom,
        "pct_positive_within_theme": row["n_positive"] / denom,
    })
    rows.append(row)

theme_sent_df_2 = pd.DataFrame(rows).sort_values("n_present", ascending=False)

# nicer formatting
theme_sent_df_2["pct_of_all_comments"] = (theme_sent_df_2["pct_of_all_comments"] * 100).round(1)
theme_sent_df_2["pct_negative_within_theme"] = (theme_sent_df_2["pct_negative_within_theme"] * 100).round(1)
theme_sent_df_2["pct_neutral_within_theme"] = (theme_sent_df_2["pct_neutral_within_theme"] * 100).round(1)
theme_sent_df_2["pct_positive_within_theme"] = (theme_sent_df_2["pct_positive_within_theme"] * 100).round(1)

theme_sent_df_2.to_csv("theme_sentiment_distribution_final.csv", index=False)
theme_sent_df_2

,theme,n_present,pct_of_all_comments,n_negative,n_neutral,n_positive,pct_negative_within_theme,pct_neutral_within_theme,pct_positive_within_theme
1,impact_on_science_and_society,227,53.5,125,6,96,55.1,2.6,42.3
0,quality_of_preprints,184,43.4,133,35,16,72.3,19.0,8.7
2,impact_on_scholars_professional_development,116,27.4,45,3,68,38.8,2.6,58.6


In [4]:
# 1) Prepare data
var = [
    "impact_on_scholars_professional_development",
    "impact_on_science_and_society",
    "quality_of_preprints",
]
theme_map = {
    "impact_on_science_and_society": "Perceived impact <br>on science & society",
    "quality_of_preprints": "Quality of preprints",
    "impact_on_scholars_professional_development": "Impact on <br> scholars' career"
}

small_neutral_threshold = 0.03  # 3% of total

for theme in var:
    df_2 = theme_sent_df_2[theme_sent_df_2["theme"] == theme].copy()
    df_2 = df_2.sort_values("theme")
    df_2["theme_display"] = df_2["theme"].astype(str).map(theme_map).fillna(df_2["theme"].astype(str))
    themes = df_2["theme_display"].tolist()

    neg = df_2["n_negative"].tolist()
    neu = df_2["n_neutral"].tolist()
    pos = df_2["n_positive"].tolist()
    totals = df_2["n_present"].tolist()


    # % labels shown inside each segment
    neg_pct = df_2["pct_negative_within_theme"].round(1).astype(str) + "%"
    neu_pct = df_2["pct_neutral_within_theme"].round(1).astype(str) + "%"
    pos_pct = df_2["pct_positive_within_theme"].round(1).astype(str) + "%"

    color_map = {
        "Negative": "#99bbe0",
        "Neutral":  "#d5e9f4",
        "Positive": "#cabad7"
    }

    # Identify small neutral segments
    neutral_frac = (df_2["n_neutral"] / df_2["n_present"].replace(0, pd.NA)).fillna(0)
    is_small_neutral = neutral_frac <= small_neutral_threshold

    # For small neutral segments, suppress bar text and use annotations instead
    neu_text_for_bar = [
        "" if small else txt
        for small, txt in zip(is_small_neutral.tolist(), neu_pct.tolist())
    ]
    
    # 2) Build stacked bar chart
    fig = go.Figure()

    fig.add_trace(go.Bar(
        y=themes,
        x=neg,
        name="Negative",
        orientation="h",
        marker_color=color_map["Negative"],
        text=neg_pct,
        textposition="inside",
        insidetextanchor="middle",
    ))

    fig.add_trace(go.Bar(
        y=themes,
        x=neu,
        name="Neutral",
        orientation="h",
        marker_color=color_map["Neutral"],
        text=neu_text_for_bar,
        textposition="inside",
        insidetextanchor="middle",
        textfont_size=13,
    ))

    fig.add_trace(go.Bar(
        y=themes,
        x=pos,
        name="Positive",
        orientation="h",
        marker_color=color_map["Positive"],
        text=pos_pct,
        textposition="inside",
        insidetextanchor="middle",
    ))

    # add total number of comments
    fig.add_trace(go.Scatter(
        y=themes,
        x=totals,
        mode="text",
        text=[f"n={t}" for t in totals],
        textposition="middle right",
        showlegend=False
    ))

    
    # Force tiny neutral labels via annotations
    # neutral segment starts at neg, ends at neg+neu -> center at neg + neu/2
    for y_label, n_neg, n_neu, txt, small in zip(
        themes, neg, neu, neu_pct.tolist(), is_small_neutral.tolist()
    ):
        if small and n_neu > 0:
            fig.add_annotation(
                x=n_neg + n_neu / 2,
                y=y_label,
                text=txt,
                showarrow=False,
                xanchor="center",
                yanchor="middle",
                font=dict(size=12, color="black"),
                textangle=0,   # force horizontal
                align="center"
            )

    
    # 3) Layout tweaks
    fig.update_layout(
        barmode="stack",
        xaxis_title="Number of comments",
        template="simple_white",
        height=180,
        width=670,
        font=dict(size=12, family="Arial"),
        legend=dict(
            x=1.06,       
            y=1.0,
            xanchor="left",
            yanchor="top"
        ),
        margin=dict(l=20, r=20, b=60, t=60),
        uniformtext_minsize=12,          
        uniformtext_mode="show"       
    )

    # update xaxis title
    fig.update_xaxes(
        tickfont=dict(size=12),
        title_font=dict(size=12)
    )

    # update yaxis title
    fig.update_yaxes(
        tickfont=dict(size=12),
        title_font=dict(size=12)
    )
    # Make text readable
    fig.update_traces(
        textfont_size=12,
        cliponaxis=False
    )

    fig.show()

    # write as PDF
    fig.write_image(f"{theme}.pdf")